# ProDy on Google Colab — A Guided Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prody/ProDy/blob/claude/prody-colab-notebook-VFYRv/docs/notebooks/ProDy_Colab_Tutorial.ipynb)

[**ProDy**](http://www.bahargroup.org/prody/) is a free, open-source Python
package for **protein structure, dynamics, and sequence analysis**. It bundles
together elastic network models (ANM, GNM), principal component analysis (PCA)
of conformational ensembles, structure parsing (PDB / mmCIF), VMD-style atom
selections, trajectory I/O, and the **Evol** suite for sequence
co-evolution — all behind one consistent Python API.

This notebook is a guided, runnable tour of the most-used ProDy modalities,
designed to work end-to-end in a fresh Google Colab runtime. Each section
opens with a short explainer that covers:

- **What it does** — the analysis or task in plain language.
- **What it needs** — the inputs you have to supply.
- **What to watch for** — limitations, pitfalls, or scaling concerns.

If you are running this on Colab, just choose **Runtime → Run all** and read
along. If you are reading on GitHub, click the badge above to launch it.


## 1. Setup & installation

**What it does:** installs ProDy and the optional 3D-viewer dependency
`py3Dmol` into the Colab runtime.

**What it needs:** an active internet connection. Colab already ships
`numpy`, `scipy`, `matplotlib`, and `biopython`, which are ProDy's only
required runtime dependencies, so the install is fast (~30 s) and never
needs `conda`.

**What to watch for:**

- Colab runtimes are ephemeral — you will need to re-run this cell whenever
  the runtime is recycled.
- A small number of advanced ProDy features (the `hpb.so` module used by
  *InSty* and *WatFinder* for hydrophobic-interaction analysis) ship as a
  precompiled binary that may not be available on every platform. None of
  the workflows in this notebook depend on it.


In [ ]:
!pip install -q prody py3Dmol


In [ ]:
import prody
import py3Dmol
import numpy as np
import matplotlib.pyplot as plt

print("ProDy version:", prody.__version__)
print("NumPy version:", np.__version__)


## 2. Parsing a protein structure

**What it does:** downloads a PDB entry by its 4-character ID (or reads a
local file) and returns an `AtomGroup` — ProDy's core container for
atomic data.

**What it needs:** either a PDB ID (network access) **or** a path to a
local `.pdb` / `.cif` file.

**What to watch for:**

- `parsePDB('1ubi')` will silently cache the file in the working directory.
- For huge assemblies, pass `subset='ca'` or `chain='A'` to keep memory low.
- For mmCIF files use `parseMMCIF` instead.


In [ ]:
ubi = prody.parsePDB('1ubi')
print(ubi)
print("Atoms:    ", ubi.numAtoms())
print("Residues: ", ubi.numResidues())
print("Chains:   ", ubi.numChains())
print("Title:    ", ubi.getTitle())


## 3. Atom selections

**What it does:** lets you carve out subsets of atoms with a VMD-like
selection grammar (`calpha`, `protein`, `resnum 1to10`, `within 5 of
chain B`, etc.). Selections return a lightweight `Selection` view, not a
copy.

**What it needs:** an `AtomGroup` and a selection string.

**What to watch for:**

- Empty selections return `None`, not an empty container — always check.
- The full grammar is documented at `prody.SELECT` and in the
  [selections reference](http://www.bahargroup.org/prody/manual/reference/atomic/select.html).


In [ ]:
calphas = ubi.select('calpha')
heavy   = ubi.select('protein and not hydrogen')
core    = ubi.select('resnum 20to50 and name CA')

print("Cα atoms in ubiquitin:", calphas.numAtoms())
print("Heavy protein atoms:  ", heavy.numAtoms())
print("Core Cα subset:       ", core.numAtoms())


## 4. Interactive 3D visualisation with py3Dmol

**What it does:** renders the protein as an interactive cartoon you can
rotate, zoom, and re-style — directly in the notebook output cell.

**What it needs:** `py3Dmol` (installed above) and an `AtomGroup`.

**What to watch for:**

- The viewer is a live JavaScript widget. If you save the notebook and
  reopen it without re-running the cell, the viewer will be blank.
- Static exports (PDF / nbviewer) likewise show nothing — re-run the cell
  in a live kernel.


In [ ]:
prody.view3D(ubi, width=500, height=400)


## 5. Anisotropic Network Model (ANM)

**What it does:** treats every Cα as a bead connected to its neighbours by
springs and diagonalises the resulting Hessian. The lowest-frequency
non-zero modes describe the **collective, functionally relevant motions** a
protein is most likely to undergo — without any MD simulation.

**What it needs:** a Cα `AtomGroup` (or any coarse-grained selection).
Optional: a cutoff distance (default 15 Å) and spring constant (`gamma`).

**What to watch for:**

- ANM predicts **directions and relative magnitudes**, not absolute
  amplitudes. Compare modes across structures, do not over-interpret raw
  numbers.
- Resolution is residue-level. For atomic detail you need MD or all-atom
  NMA.
- The first 6 modes are zero (rigid-body translations + rotations) and
  are skipped by default.


In [ ]:
anm = prody.ANM('ubiquitin')
anm.buildHessian(calphas, cutoff=15.0)
anm.calcModes(n_modes=20)

print(anm)
print("First non-zero mode eigenvalue:", anm[0].getEigval())
print("Collectivity of mode 1:", prody.calcCollectivity(anm[0]))


### Square fluctuations and cross-correlations

Two staple plots fall right out of an ANM calculation:

- **Square fluctuations** per residue: which parts of the chain are most
  mobile under collective motion?
- **Cross-correlation map**: which residue pairs move together (red) or
  in opposition (blue)?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plt.sca(axes[0])
prody.showSqFlucts(anm[:5])
axes[0].set_title("ANM square fluctuations (first 5 modes)")

plt.sca(axes[1])
prody.showCrossCorr(anm)
axes[1].set_title("ANM cross-correlation map")

plt.tight_layout()
plt.show()


### Animating the slowest mode in 3D

`view3D` accepts a `mode=` argument and will draw arrows or animate the
trajectory along that mode.


In [ ]:
prody.view3D(calphas, mode=anm[0], scale=80, width=500, height=400)


## 6. Gaussian Network Model (GNM)

**What it does:** the isotropic, scalar cousin of ANM. Each residue gets a
single mobility value per mode — cheaper to compute and often enough for
identifying flexible regions and hinge sites.

**What it needs:** the same Cα selection. Default cutoff is 10 Å.

**What to watch for:**

- GNM gives **magnitudes only** — no directions. Use ANM if you want to
  animate motion in 3D.
- Mode 1 (the slowest non-zero mode) typically separates the protein into
  two anti-correlated dynamic domains; sign flips between residues mark
  hinge points.


In [ ]:
gnm = prody.GNM('ubiquitin')
gnm.buildKirchhoff(calphas, cutoff=10.0)
gnm.calcModes(n_modes=20)
print(gnm)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plt.sca(axes[0])
prody.showMode(gnm[0])
axes[0].set_title("GNM mode 1 (slowest) — sign flips mark hinges")

plt.sca(axes[1])
prody.showContactMap(gnm)
axes[1].set_title("Cα contact map (Kirchhoff)")

plt.tight_layout()
plt.show()


## 7. Principal Component Analysis on an NMR ensemble

**What it does:** extracts the dominant **observed** motions from a set of
related conformations — multiple NMR models, an MD trajectory, or
homologous X-ray structures.

**What it needs:** an `Ensemble` of structurally aligned conformers with
the same number of atoms.

**What to watch for:**

- Conformations must be aligned first (`ensemble.superpose()`); ProDy can
  do this for you.
- PCA quality scales with ensemble size and diversity. A handful of nearly
  identical structures will give noisy modes.
- PCA modes from experiments and ANM modes from theory often agree well —
  Section 8 quantifies this.

We use **PDB 2K39**, an NMR ensemble of ubiquitin with 116 models.


In [ ]:
ubi_nmr = prody.parsePDB('2k39', subset='ca')
print("Models in 2K39:", ubi_nmr.numCoordsets())

ensemble = prody.Ensemble('2K39 Cα ensemble')
ensemble.setCoords(ubi_nmr.getCoords())
ensemble.addCoordset(ubi_nmr.getCoordsets())
ensemble.setAtoms(ubi_nmr)
ensemble.superpose()

pca = prody.PCA('2K39')
pca.buildCovariance(ensemble)
pca.calcModes(n_modes=20)
print(pca)
print("Variance fraction (mode 1):", prody.calcFractVariance(pca[0]))


In [ ]:
plt.figure(figsize=(8, 4))
prody.showFractVars(pca[:10])
prody.showCumulFractVars(pca[:10])
plt.title("Variance explained by PCA modes (2K39 NMR ensemble)")
plt.show()


## 8. Comparing experimental (PCA) vs theoretical (ANM) motion

**What it does:** computes the overlap between PCA modes (what the
ensemble actually does) and ANM modes (what the elastic network predicts).
A diagonal-heavy overlap table is the classic sanity check that ENM-based
dynamics agree with experiment.

**What it needs:** two `ModeSet` objects of compatible dimension.

**What to watch for:** the ANM and PCA must be calculated on the **same
selection** (here, ubiquitin Cα). Re-running ANM on the NMR Cα selection
keeps the comparison clean.


In [ ]:
anm_2k39 = prody.ANM('2K39 ANM')
anm_2k39.buildHessian(ubi_nmr)
anm_2k39.calcModes(n_modes=20)

prody.printOverlapTable(pca[:5], anm_2k39[:5])


## 9. Working with DCD trajectories

**What it does:** ProDy reads and writes CHARMM/NAMD **DCD** trajectory
files and feeds them straight into PCA/EDA workflows.

**What it needs:** a `.dcd` file plus the matching topology (PDB).

**What to watch for:**

- Only DCD is native. For `.xtc` / `.trr` (GROMACS) or `.nc` (AMBER), use
  [MDAnalysis](https://www.mdanalysis.org/) or
  [mdtraj](https://www.mdtraj.org/) and pass the coordinates into a
  `prody.Ensemble`.
- Big trajectories should be processed frame-by-frame
  (`for frame in trajectory:`) instead of loading every coordinate set
  into RAM.

The cell below shows the **API only** — running it requires you to
provide your own `topology.pdb` + `trajectory.dcd`:


In [ ]:
# Example only — uncomment and point at your own files to run.
#
# topology   = prody.parsePDB('topology.pdb')
# trajectory = prody.Trajectory('trajectory.dcd')
# trajectory.link(topology)
# trajectory.setAtoms(topology.calpha)
#
# eda = prody.EDA('MD essential dynamics')
# eda.buildCovariance(trajectory)
# eda.calcModes(n_modes=20)
# print(eda)


## 10. Sequence analysis with Evol

**What it does:** ProDy's **Evol** subpackage downloads multiple sequence
alignments from Pfam and computes per-residue conservation (Shannon
entropy) and pairwise co-evolution (mutual information). These signals
often line up with mechanically important residues found by ANM/GNM.

**What it needs:** a Pfam family accession (e.g. `PF00240` for
ubiquitin) **or** a UniProt ID to search Pfam with.

**What to watch for:**

- Pfam and InterPro APIs change occasionally; if the fetch fails the rest
  of the notebook is unaffected — skip this section.
- MSAs for popular families can be large (tens of MB).


In [ ]:
try:
    msa_path = prody.fetchPfamMSA('PF00240', alignment='seed')
    msa = prody.parseMSA(msa_path)
    msa_refined = prody.refineMSA(msa, rowocc=0.8, seqid=0.98)
    entropy = prody.calcShannonEntropy(msa_refined)

    plt.figure(figsize=(10, 3))
    plt.plot(entropy)
    plt.xlabel("Alignment column")
    plt.ylabel("Shannon entropy")
    plt.title("Conservation across the ubiquitin Pfam family (PF00240)")
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print("Pfam fetch unavailable in this runtime:", exc)


## 11. Saving and downloading results

**What it does:** ProDy persists models in a few formats:

- `saveModel()` — pickled `.npz` of an ANM/GNM/PCA result you can reload
  later with `loadModel()`.
- `writeNMD()` — a text format read by **NMWiz**, the VMD plugin for
  visualising normal modes.
- `writePDB()` — standard PDB output.

**What to watch for:** Colab's filesystem is wiped when the runtime is
recycled. Use `google.colab.files.download(...)` to pull files to your
local machine, or mount Google Drive.


In [ ]:
prody.saveModel(anm, 'ubi_anm')
prody.writeNMD('ubi_anm.nmd', anm[:5], calphas)
prody.writePDB('ubi_ca.pdb', calphas)

import os
for f in ['ubi_anm.anm.npz', 'ubi_anm.nmd', 'ubi_ca.pdb']:
    if os.path.exists(f):
        print(f"{f}\t{os.path.getsize(f):,} bytes")


In [ ]:
# Uncomment to download the saved files locally when running on Colab:
#
# from google.colab import files
# files.download('ubi_anm.nmd')
# files.download('ubi_ca.pdb')


## 12. Where to go next

This notebook covered the core ProDy modalities. The package has many
more, all reachable from the top-level `prody.` namespace:

- **ClustENM / ClustENMD** — hybrid ENM + MD sampling
  (`prody.dynamics.clustenm`).
- **Perturbation Response Scanning (PRS)** — allosteric communication maps
  (`prody.calcPerturbResponse`).
- **InSty** — protein–protein interaction analysis
  (`prody.proteins.interactions`).
- **WatFinder** — water-bridge networks
  (`prody.proteins.waterbridges`).
- **ChromDy** — chromatin dynamics from Hi-C data
  (`prody.chromatin`).
- **Spectrus** — dynamical domain decomposition
  (`prody.domain_decomposition`).

**Resources**

- Tutorials — <http://www.bahargroup.org/prody/tutorials>
- API reference — <http://www.bahargroup.org/prody/manual>
- NMWiz (VMD plugin for normal modes) —
  <http://www.bahargroup.org/prody/nmwiz>
- Source / issues — <https://github.com/prody/ProDy>

If you use ProDy in published work, please cite the *Bioinformatics*
papers listed in the [README](https://github.com/prody/ProDy#readme).
